In [18]:
import pandas as pd
import torch
import os
file_path = r'data\rawData'
txt_file_path = r'data\processedData'
new_path = r'd:\Desktop\PHD\reasearch\biyework\maml'
os.chdir(new_path)
# 检查当前路径是否切换成功
current_path = os.getcwd()
print(current_path)

d:\Desktop\PHD\reasearch\biyework\maml


In [23]:

# 读取CSV文件
csv_file_path = r'data\processedData\all_data_new.csv'  # 替换为你的CSV文件路径
df = pd.read_csv(csv_file_path)

# 读取 location_vector.csv 文件
location_vector_path = r'model\v1\output\location_vector.csv'  # 替换为你的location_vector.csv文件路径
location_df = pd.read_csv(location_vector_path)

# 创建 location_id 到 idx 的映射
location_id_to_idx = dict(zip(location_df['location_id'], location_df['idx']))
print(df['location_id'].unique())
print(location_id_to_idx)
# 将 location_id 转换为 idx，并打印0-36缺少的点
df['location_id'] = df['location_id'].map(location_id_to_idx)

# 打印转换后的 location_id 列，检查是否有丢失的点
print("Mapped Location IDs:")
print(len(df['location_id'].unique()))
print(df['location_id'].unique())
# 检查并转换DataFrame中的数据类型
df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

data_dict = {
    'rssi': torch.tensor(df[['rssi','average_rssi',"snr","average_snr"]].values, dtype=torch.float32),
    'snr': torch.tensor(df[['snr',"average_snr","rssi",'average_rssi']].values, dtype=torch.float32),
    'sf': torch.tensor(df['sf'].values, dtype=torch.float32),
    'tp': torch.tensor(df['tp'].values, dtype=torch.float32),
    'label':  torch.tensor(df['location_id'].values, dtype=torch.int64)
}
# 把数据变成24915/16 2 16 的shape
# 确保数据长度是 batch_size * 2 * 16 的倍数
batch_size = 16  # 你可以根据需要调整 batch_size
total_length = len(df)
required_length = (total_length // (batch_size * 2 * 16)) * (batch_size * 2 * 16)

# 截断数据以匹配所需长度
data_dict['rssi'] = data_dict['rssi'][:required_length]
data_dict['snr'] = data_dict['snr'][:required_length]
data_dict['sf'] = data_dict['sf'][:required_length]
data_dict['tp'] = data_dict['tp'][:required_length]
data_dict['label'] = data_dict['label'][:required_length]


# 保存为PTH文件
pth_file_path = r"model\v1\input\FLOOR3.pth"  
# pth的形式以字典的方式保存rssi,snr,sf,tp,location_id,label
torch.save(data_dict, pth_file_path)

['302' '306-304' '308' '310' '312' '316' '318' '320' '322' '324' '326'
 '328' '330' '332' '334' '336' '338' '340' '342' '344' '346' '348' '350'
 '352' '354' '356' '358' '360' '362' '364' '366' '368' '370' '372'
 'point1' 'point2' 'point3']
{'point1': 0, '372': 1, '370': 2, '368': 3, '366': 4, '364': 5, '362': 6, '360': 7, '358': 8, '356': 9, 'point2': 10, '354': 11, '352': 12, '350': 13, '348': 14, '346': 15, '344': 16, '342': 17, '340': 18, '338': 19, '336': 20, '334': 21, 'point3': 22, '332': 23, '330': 24, '328': 25, '326': 26, '324': 27, '322': 28, '320': 29, '318': 30, '316': 31, '312': 32, '310': 33, '308': 34, '306-304': 35, '302': 36}
Mapped Location IDs:
37
[36 35 34 33 32 31 30 29 28 27 26 25 24 23 21 20 19 18 17 16 15 14 13 12
 11  9  8  7  6  5  4  3  2  1  0 10 22]


In [22]:
# 验证一下这段代码
# 读取PTH文件
data_tensor = torch.load(r"model\v1\input\FLOOR3.pth", weights_only=True)
# 打印数据
print(data_tensor)
print(data_tensor['rssi'].shape)
print(data_tensor['snr'].shape)
print(data_tensor['sf'].shape)
print(data_tensor['tp'].shape)
print(data_tensor['label'].shape)

{'rssi': tensor([[-116., -124.,  -11.],
        [-117., -124.,  -11.],
        [-117., -124.,  -11.],
        ...,
        [ -81.,  -79.,    6.],
        [ -79.,  -79.,    6.],
        [ -77.,  -79.,    6.]]), 'snr': tensor([[ -11.,  -11., -116.],
        [ -11.,  -11., -117.],
        [ -11.,  -11., -117.],
        ...,
        [   6.,    6.,  -81.],
        [   6.,    6.,  -79.],
        [   6.,    6.,  -77.]]), 'sf': tensor([9., 9., 9.,  ..., 9., 9., 9.]), 'tp': tensor([2., 2., 2.,  ..., 2., 2., 2.]), 'label': tensor([36, 36, 36,  ..., 22, 22, 22])}
torch.Size([24576, 3])
torch.Size([24576, 3])
torch.Size([24576])
torch.Size([24576])
torch.Size([24576])
